In [1]:
import os
from glob import glob
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Step 1: Load all .md files
md_dir = "./md_pages"
md_files = glob(os.path.join(md_dir, "*.md"))

documents = []
doc_ids = []
for filepath in md_files:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
        documents.append(content)
        doc_ids.append(os.path.basename(filepath))

# Step 2: Embed docs
model = SentenceTransformer("jhgan/ko-sroberta-multitask")  # Kor/En best practice, change as needed
embeddings = model.encode(documents, show_progress_bar=True, convert_to_numpy=True)

# Step 3: Upload to Qdrant
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION_NAME = "heum_md_docs"

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

# Create collection (if not exists)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE)
)

# Prepare points
points = [
    PointStruct(
        id=idx,
        vector=embeddings[idx],
        payload={
            "filename": doc_ids[idx],
            "text": documents[idx][:1000]  # Preview of content
        }
    ) for idx in range(len(documents))
]
# Batch upload
BATCH = 64
for i in tqdm(range(0, len(points), BATCH)):
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i + BATCH]
    )

print(f"✅ Uploaded {len(documents)} documents to Qdrant collection '{COLLECTION_NAME}'.")

KeyboardInterrupt: 